Before you turn this problem in, make sure everything runs as expected. First, **restart the kernel** (in the menubar, select Kernel$\rightarrow$Restart) and then **run all cells** (in the menubar, select Cell$\rightarrow$Run All).

Make sure you fill in any place that says `YOUR CODE HERE` or "YOUR ANSWER HERE", as well as your name and collaborators below:

In [ ]:
NAME = ""
COLLABORATORS = ""

---

## 3.1 Backtest a simple trading strategy 

**Created by: Pengfei Zhao**  
**© 2025 Investment Practice Course**

* A `backtest` is a historical analysis that evaluates the performance of a trading or investment strategy using historical market data. The primary purpose of a backtest is to assess how well a given trading strategy or set of trading rules would have performed in the past under specific historical market conditions. 

* Backtesting is a crucial step in the development and validation of trading strategies and is commonly used in finance and investment management.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

### 1. Load and prepare Microsoft (MSFT) stock data

In [ ]:
# Load MSFT stock data
msft = pd.read_csv('MSFT.csv', index_col='Date', parse_dates=True)
msft = msft[['Close']]
msft.head()

### 2. Strategy Implementation Method 1: Transaction-Based Approach (More Intuitive)

This method tracks actual buy/sell transactions:
- Buy when short MA crosses **above** long MA
- Sell when short MA crosses **below** long MA
- Hold position between signals

In [ ]:
# Strategy parameters
short_window = 10  # MA10
long_window = 50   # MA50
initial_cash = 100000

# Calculate moving averages
msft['MA10'] = msft['Close'].rolling(window=short_window).mean()
msft['MA50'] = msft['Close'].rolling(window=long_window).mean()

# Remove NaN values
msft = msft.dropna()

# Initialize position tracking
msft['Position'] = 0
cash = initial_cash
total_profit = 0

# Implement transaction-based strategy
for i in range(1, len(msft)):
    if msft['MA10'].iloc[i] > msft['MA50'].iloc[i] and msft['Position'].iloc[i-1] == 0:
        # Buy signal: MA10 crosses above MA50 and no position is open
        shares_to_buy = 1
        msft.iloc[i, msft.columns.get_loc('Position')] = shares_to_buy
        buy_price = msft['Close'].iloc[i]
        cash -= shares_to_buy * buy_price
        print(f'Buy at {msft.index[i].date()} with price ${buy_price:.2f}')
        
    elif msft['MA10'].iloc[i] < msft['MA50'].iloc[i] and msft['Position'].iloc[i-1] > 0:
        # Sell signal: MA10 crosses below MA50 and a position is open
        sell_price = msft['Close'].iloc[i]
        cash += msft['Position'].iloc[i-1] * sell_price
        profit = (sell_price - buy_price) * msft['Position'].iloc[i-1]
        total_profit += profit
        print(f'Sell at {msft.index[i].date()} with price ${sell_price:.2f}, Profit: ${profit:.2f}')
        msft.iloc[i, msft.columns.get_loc('Position')] = 0
        
    else:
        # Hold current position
        msft.iloc[i, msft.columns.get_loc('Position')] = msft['Position'].iloc[i-1]

# Calculate daily profit for visualization
msft['Position_Lag'] = msft['Position'].shift(1).fillna(0)
msft['Close_Prev'] = msft['Close'].shift(1)
msft['Daily_Profit'] = (msft['Close'] - msft['Close_Prev']) * msft['Position_Lag']
msft['Cumulative_Profit'] = msft['Daily_Profit'].cumsum()

# Calculate final portfolio value
final_value = cash + (msft['Close'].iloc[-1] * msft['Position'].iloc[-1])
print(f'\n=== Results ===')
print(f'Initial cash: ${initial_cash:.2f}')
print(f'Final portfolio value: ${final_value:.2f}')
print(f'Total profit: ${total_profit:.2f}')

### 3. Visualize the cumulative profit over time

In [ ]:
# Plot cumulative profit
plt.figure(figsize=(12, 6))
msft['Cumulative_Profit'].plot()
plt.axhline(y=0, color='red', linestyle='--', alpha=0.7)
plt.title(f'Cumulative Profit Over Time\nTotal Profit: ${total_profit:.2f}')
plt.xlabel('Date')
plt.ylabel('Cumulative Profit ($)')
plt.grid(True, alpha=0.3)
plt.show()